# DIN ranking on Amazon Video_Games

Trains a Deep Interest Network (DIN) on next-item ranking over the Amazon Reviews 2023 `Video_Games` category, using the `amazon_ranking` data module (leakage-free negative sampling + cached eval candidates). Reports Recall@k / NDCG@k / MRR@k and sampled AUC on the test split.

Runs on a Colab T4 GPU via papermill. By default it subsamples users (`MAX_USERS`) so a baseline finishes quickly; set `MAX_USERS=None` for the full category.

In [ ]:
import os
try:
    from google.colab import drive  # type: ignore
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print('IN_COLAB:', IN_COLAB)

In [ ]:
import sys, os
# Locate the repo root so `tiger_semantic_id` and `amazon_ranking` are importable.
CANDIDATES = [
    '/content/drive/MyDrive/colab/recsys_playground/recsys_playground',
    '/content/recsys_playground',
    os.getcwd(),
    os.path.dirname(os.path.dirname(os.getcwd())),
]
REPO_ROOT = next(
    (p for p in CANDIDATES if os.path.exists(os.path.join(p, 'amazon_ranking'))
     and os.path.exists(os.path.join(p, 'tiger_semantic_id'))),
    os.getcwd(),
)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print('REPO_ROOT:', REPO_ROOT)

In [ ]:
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(cpu)')

# --- Data / experiment config ---
DATASET_NAME = 'Video_Games'
DATA_DIR = '/content/drive/MyDrive/colab/data/amazon_ranking'
RESULTS_DIR = os.path.join(REPO_ROOT, 'experiments/20260530_amazon_din')
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

MAX_HIST_LEN = 20
MIN_USER_INTERACTIONS = 5
N_EVAL_NEGATIVES = 100
N_TRAIN_NEGATIVES = 4
MAX_USERS = 50000   # subsample users for a tractable T4 baseline; set None for full data
SEED = 42

# --- DIN / training config ---
EMBED_DIM = 32
EPOCHS = 3
BATCH_SIZE = 2048
LR = 1e-3
print('config ready')

In [ ]:
import subprocess
REVIEWS_URL = 'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Video_Games.jsonl.gz'
reviews_path = os.path.join(DATA_DIR, 'Video_Games.jsonl.gz')
if not os.path.exists(reviews_path):
    print('Downloading', REVIEWS_URL)
    subprocess.run(['wget', '-q', '-O', reviews_path, REVIEWS_URL], check=True)
print('reviews_path:', reviews_path, round(os.path.getsize(reviews_path) / 1e6, 1), 'MB')

In [ ]:
import numpy as np
import pandas as pd
from tiger_semantic_id.src.data import load_reviews_df

reviews = load_reviews_df(reviews_path, dataset_format='2023')
print('raw reviews:', reviews.shape, 'users:', reviews['user_id'].nunique())
if MAX_USERS is not None:
    rng = np.random.default_rng(SEED)
    users = reviews['user_id'].unique()
    if len(users) > MAX_USERS:
        keep = set(rng.choice(users, size=MAX_USERS, replace=False).tolist())
        reviews = reviews[reviews['user_id'].isin(keep)].copy()
print('reviews used:', reviews.shape, 'users:', reviews['user_id'].nunique())

In [ ]:
from amazon_ranking.src.datamodule import DataModuleConfig, SequenceRankingDataModule

dm_cfg = DataModuleConfig(
    max_hist_len=MAX_HIST_LEN,
    min_user_interactions=MIN_USER_INTERACTIONS,
    n_eval_negatives=N_EVAL_NEGATIVES,
    n_train_negatives=N_TRAIN_NEGATIVES,
    neg_strategy='uniform',
    seed=SEED,
)
dm = SequenceRankingDataModule.from_reviews(reviews, dm_cfg)
dm.build()
train_ex = dm.train_examples()
print('num_items:', dm.num_items, 'num_users:', dm.num_users)
print('train examples:', len(train_ex))
print('eval users (test):', len(dm.eval_examples('test')))

In [ ]:
import time
from amazon_ranking.src.din import DIN, DINTrainConfig, train_din, evaluate_ranking

model = DIN(num_items=dm.num_items, embed_dim=EMBED_DIM)
train_cfg = DINTrainConfig(embed_dim=EMBED_DIM, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, seed=SEED)
t0 = time.time()
train_hist = train_din(model, train_ex, max_hist_len=MAX_HIST_LEN, cfg=train_cfg, device=DEVICE)
train_secs = time.time() - t0
print('train losses:', train_hist)
print('train wall secs:', round(train_secs, 1))

In [ ]:
t0 = time.time()
metrics = evaluate_ranking(model, dm.eval_examples('test'), max_hist_len=MAX_HIST_LEN, ks=(5, 10), device=DEVICE)
eval_secs = time.time() - t0
print('eval wall secs:', round(eval_secs, 1))
for key in sorted(metrics):
    print(f'{key}: {metrics[key]:.4f}')

In [ ]:
import json, datetime
result = {
    'dataset': DATASET_NAME,
    'model': 'DIN',
    'config': {
        'max_hist_len': MAX_HIST_LEN, 'min_user_interactions': MIN_USER_INTERACTIONS,
        'n_eval_negatives': N_EVAL_NEGATIVES, 'n_train_negatives': N_TRAIN_NEGATIVES,
        'max_users': MAX_USERS, 'embed_dim': EMBED_DIM, 'epochs': EPOCHS,
        'batch_size': BATCH_SIZE, 'lr': LR, 'seed': SEED,
    },
    'data': {'num_users': dm.num_users, 'num_items': dm.num_items, 'train_examples': len(train_ex)},
    'train': train_hist,
    'train_secs': train_secs,
    'eval_secs': eval_secs,
    'metrics': metrics,
    'device': DEVICE,
    'timestamp': datetime.datetime.now().isoformat(timespec='seconds'),
}
out_path = os.path.join(RESULTS_DIR, 'din_video_games_results.json')
with open(out_path, 'w') as f:
    json.dump(result, f, indent=2)
print('saved:', out_path)
print(json.dumps(result['metrics'], indent=2))